# 05. Analiza zdarzeń wokół Binance i wolumen względny

Model z BTC szacuje oczekiwany zwrot BNB. Dla każdego zdarzenia liczymy CAR w jednostkach zwrotu logarytmicznego, statystykę z błędem prognozy regresji i dwustronne wartości $p$, korygowane metodą Holma dla 16 zaplanowanych porównań. W analizie wolumenu łączymy względny obrót BNB i wielkość jego dziennego zwrotu w wybranym okresie. Zasady określają rozdziały 9.3 i 10 dokumentu metod.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from market_data import daily_price_ratios, download


def event_study(returns, event_date, excluded_dates, win_before=1, win_after=1, est_before=120, est_after=11):
    event_ts = pd.Timestamp(event_date, tz="UTC")
    est_start = event_ts - pd.Timedelta(days=est_before)
    est_end = event_ts - pd.Timedelta(days=est_after)
    estimation = returns.loc[est_start:est_end].dropna()
    estimation = estimation.loc[~estimation.index.isin(excluded_dates)]
    length = len(estimation)
    result = {"L": length, "alpha": np.nan, "beta": np.nan, "s_eps": np.nan, "AR": None,
              "CAR": np.nan, "t": np.nan, "p": np.nan, "CAR_alpha0": np.nan}
    if length < 3:
        return {**result, "status": "za mało danych w oknie estymacji"}
    m = estimation.btc
    r = estimation.bnb
    centered_m = m - m.mean()
    sxx = (centered_m ** 2).sum()
    if sxx == 0:
        return {**result, "status": "zerowa wariancja BTC w oknie estymacji"}
    beta = (centered_m * (r - r.mean())).sum() / sxx
    alpha = r.mean() - beta * m.mean()
    residuals = r - (alpha + beta * m)
    dof = length - 2
    s_eps = np.sqrt((residuals ** 2).sum() / dof)
    result.update(alpha=alpha, beta=beta, s_eps=s_eps)

    event_days = pd.date_range(event_ts - pd.Timedelta(days=win_before),
                               event_ts + pd.Timedelta(days=win_after), freq="D")
    event = returns.reindex(event_days)
    missing = event.isna().any(axis=1).sum()
    if missing:
        return {**result, "status": f"brak danych w oknie zdarzenia ({missing} dni)"}

    ar = event.bnb - (alpha + beta * event.btc)
    car = ar.sum()
    result.update(AR=ar, CAR=car, CAR_alpha0=(event.bnb - beta * event.btc).sum())
    if s_eps == 0:
        return {**result, "status": "zerowa wariancja reszt w oknie estymacji"}

    size = len(event)
    market_deviation = (event.btc - m.mean()).sum()
    car_se = s_eps * np.sqrt(size + size ** 2 / length + market_deviation ** 2 / sxx)
    t = car / car_se
    status = "krótka próba estymacji" if length < 30 else "ok"
    return {**result, "t": t, "p": 2 * stats.t.sf(abs(t), df=dof), "status": status}


def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    finite = np.flatnonzero(np.isfinite(p_values))
    order = finite[np.argsort(p_values[finite], kind="stable")]
    adjusted = np.full(p_values.shape, np.nan)
    raw = p_values[order] * (len(p_values) - np.arange(len(order)))
    adjusted[order] = np.minimum(1, np.maximum.accumulate(raw))
    return adjusted


def relative_volume(quote_volume, window=30):
    calendar = quote_volume.sort_index().asfreq("D").astype(float)
    prior_mean = calendar.shift(1).rolling(window).mean()
    return np.log(calendar.where(calendar > 0) / prior_mean.where(prior_mean > 0))

In [2]:
# Przykłady z dokumentu dane-testowe-rozszerzone.md
dates = pd.date_range("2023-01-01", periods=19, freq="D", tz="UTC")
m_values = [0.01, -0.02, 0.03, -0.01, 0.02] + [0.0] * 10 + [0.01, -0.02, 0.00, 0.0]
r_values = [0.02, -0.03, 0.05, -0.01, 0.03] + [0.0] * 10 + [-0.01, -0.08, 0.01, 0.0]
example = pd.DataFrame({"bnb": r_values, "btc": m_values}, index=dates)
control = dict(event_date="2023-01-17", excluded_dates=set(), est_before=16, est_after=12)
result = event_study(example, **control)
assert result["status"] == "krótka próba estymacji" and result["L"] == 5
np.testing.assert_allclose(result["AR"], [-0.0281395349, -0.0520930233, 0.0072093023], atol=1e-9)
keys = ["alpha", "beta", "s_eps", "CAR", "CAR_alpha0", "t", "p"]
expected = [0.0027906977, 1.5348837209, 0.0030499714, -0.0730232558,
            -0.0646511628, -10.4434846730, 0.0018740542]
np.testing.assert_allclose([result[k] for k in keys], expected, atol=1e-8)
missing = event_study(example.drop(dates[16]), **control)
assert missing["status"].startswith("brak danych") and np.isnan(missing["CAR"])
np.testing.assert_allclose(holm_adjust([0.03, 0.01, np.nan, 0.02]), [0.06, 0.04, np.nan, 0.06])
assert np.isnan(holm_adjust([np.nan])).all()

volumes = pd.Series([100.0, 200.0, 300.0, 400.0], index=dates[:4])
np.testing.assert_allclose(relative_volume(volumes, 3).iloc[-1], np.log(2))
assert np.isnan(relative_volume(volumes.drop(dates[2]), 2).iloc[-1])
volumes.iloc[0] = 0.0
np.testing.assert_allclose(relative_volume(volumes, 3).iloc[-1], np.log(2.4))
volumes.iloc[-1] = 0.0
assert np.isnan(relative_volume(volumes, 3).iloc[-1])
assert np.isnan(relative_volume(pd.Series([0.0, 0.0, 1.0], index=dates[:3]), 2).iloc[-1])
print("Obliczenia kontrolne dla analizy zdarzeń, korekty Holma i wolumenu przeszły pomyślnie.")

Obliczenia kontrolne dla analizy zdarzeń, korekty Holma i wolumenu przeszły pomyślnie.


In [3]:
now = pd.Timestamp.now(tz="UTC")
bnb_df, bnb_checks = download("BNBUSDT", now=now)
btc_df, btc_checks = download("BTCUSDT", now=now)
for symbol, candles, checks in (("BNBUSDT", bnb_df, bnb_checks), ("BTCUSDT", btc_df, btc_checks)):
    print(f"{symbol}: ostatni dzień UTC {candles.index.max().date()}, brakujące dni: {len(checks['missing_dates'])}")

returns = pd.DataFrame({"bnb": np.log(daily_price_ratios(bnb_df.close)),
                        "btc": np.log(daily_price_ratios(btc_df.close))}).sort_index()

events = [
    ("2023-03-27", "Pozew CFTC przeciw Binance i CZ"),
    ("2023-06-05", "Pozew SEC przeciw Binance i CZ"),
    ("2023-11-21", "Ugoda Binance z władzami USA i odejście CZ"),
    ("2024-04-30", "Wyrok 4 mies. więzienia dla CZ"),
    ("2024-09-27", "Wyjście CZ na wolność"),
    ("2025-05-29", "Zakończenie sprawy SEC przeciw Binance"),
    ("2025-10-23", "Ułaskawienie CZ"),
    ("2026-10-29", "Atak na Binance"),
]
excluded = set(bnb_checks["outliers"].index) | set(btc_checks["outliers"].index)
for date, _ in events:
    center = pd.Timestamp(date, tz="UTC")
    excluded.update(pd.date_range(center - pd.Timedelta(days=5), center + pd.Timedelta(days=5), freq="D"))

rows = []
for date, name in events:
    for days in (1, 5):
        result = event_study(returns, date, excluded, win_before=days, win_after=days)
        rows.append({"Data UTC": date, "Zdarzenie": name, "Okno": f"[-{days},+{days}]", "L": result["L"],
                     "CAR": result["CAR"], "CAR przy α=0": result["CAR_alpha0"],
                     "t": result["t"], "p": result["p"], "Status": result["status"]})
summary = pd.DataFrame(rows)
summary.insert(summary.columns.get_loc("p") + 1, "p Holma", holm_adjust(summary["p"]))
print(summary.to_string(index=False, na_rep="brak", float_format=lambda value: f"{value:.6f}"))

BNBUSDT: ostatni dzień UTC 2026-11-15, brakujące dni: 0
BTCUSDT: ostatni dzień UTC 2026-11-15, brakujące dni: 0
  Data UTC                                  Zdarzenie    Okno   L       CAR  CAR przy α=0         t        p  p Holma Status
2023-03-27            Pozew CFTC przeciw Binance i CZ [-1,+1] 110 -0.015589     -0.023099 -0.481859 0.630881 1.000000     ok
2023-03-27            Pozew CFTC przeciw Binance i CZ [-5,+5] 110 -0.042288     -0.069823 -0.659877 0.510737 1.000000     ok
2023-06-05             Pozew SEC przeciw Binance i CZ [-1,+1]  99 -0.087436     -0.090355 -3.315602 0.001287 0.019310     ok
2023-06-05             Pozew SEC przeciw Binance i CZ [-5,+5]  99 -0.210169     -0.220871 -3.992607 0.000127 0.002034     ok
2023-11-21 Ugoda Binance z władzami USA i odejście CZ [-1,+1] 110 -0.041018     -0.043445 -2.062924 0.041518 0.581257     ok
2023-11-21 Ugoda Binance z władzami USA i odejście CZ [-5,+5] 110 -0.075549     -0.084447 -1.915573 0.058064 0.754831     ok
2024-04-30   

In [4]:
# Historia obrotu służy do uzupełnienia pierwszych okien względnego wolumenu
period = bnb_df.loc["2023-01-01":]
v_rel = relative_volume(bnb_df.quote_volume).reindex(period.index)
abs_r = np.log(daily_price_ratios(period.close)).abs()
volume_sample = pd.DataFrame({"abs_r": abs_r, "v_rel": v_rel}).dropna()
n = len(volume_sample)
print(f"Wolumen względny a wielkość dziennego zwrotu BNB (n = {n}):")
if n < 2:
    print("  Brak dwóch wspólnych obserwacji do obliczenia korelacji.")
elif volume_sample.abs_r.nunique() < 2 or volume_sample.v_rel.nunique() < 2:
    print("  Korelacja jest nieokreślona, ponieważ co najmniej jeden szereg jest stały.")
else:
    print(f"  Pearson:  {volume_sample.abs_r.corr(volume_sample.v_rel):.6f}")
    print(f"  Spearman: {volume_sample.abs_r.corr(volume_sample.v_rel, method='spearman'):.6f}")
    if n < 90:
        print("  Próba poniżej 90 wspólnych obserwacji.")

Wolumen względny a wielkość dziennego zwrotu BNB (n = 1414):
  Pearson:  0.471459
  Spearman: 0.401984
